In [1]:
import os
import sys
import pyspark

print("Python:", sys.version)
print("PySpark:", pyspark.__version__)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

Python: 3.14.2 (main, Oct 28 2026, 19:03:49) [GCC 13.3.0]
PySpark: 3.5.6
JAVA_HOME: /usr/local/sdkman/candidates/java/current


In [2]:
import subprocess

subprocess.run(["java", "-version"])

openjdk version "25.0.5.0.2" 2026-10-22 LTS
OpenJDK Runtime Environment Microsoft-15804669 (build 25.0.5.0.2+1-LTS)
OpenJDK 64-Bit Server VM Microsoft-15804669 (build 25.0.5.0.2+1-LTS, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

In [3]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
)

print(os.environ["JAVA_HOME"])

/usr/lib/jvm/java-17-openjdk-amd64


In [4]:
import os
import subprocess

print("JAVA_HOME:", os.environ["JAVA_HOME"])

subprocess.run(["java", "-version"])

JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64


openjdk version "17.0.21.0.3" 2026-10-24
OpenJDK Runtime Environment (build 17.0.21.0.3+1-1-24.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.0.3+1-1-24.04.1-Ubuntu, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

In [5]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkTest")
    .master("local[*]")
    .getOrCreate()
)

spark.range(5).show()

26/11/18 11:23:19 WARN Utils: Your hostname, codespaces-c9dfa0 resolves to a loopback address: 127.0.0.1; using 10.0.10.123 instead (on interface eth0)
26/11/18 11:23:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 11:23:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/18 11:23:21 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [5]:
spark.sql("SHOW DATABASES").show()


NameError: name 'spark' is not defined

In [8]:
spark.stop()

In [6]:
from pyspark.sql import SparkSession

# Initialize Spark session with Iceberg configurations
spark = SparkSession.builder \
  .appName("IcebergLocalDevelopment") \
  .config('spark.jars.packages', 'org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2') \
  .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
  .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
  .config("spark.sql.catalog.local.type", "hadoop") \
  .config("spark.sql.catalog.local.warehouse", "spark-warehouse/iceberg") \
  .getOrCreate()


26/11/18 11:26:29 WARN Utils: Your hostname, codespaces-c9dfa0 resolves to a loopback address: 127.0.0.1; using 10.0.10.123 instead (on interface eth0)
26/11/18 11:26:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/workspaces/learn_icebergy/iceberg_pyspark_project/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/codespace/.ivy2/cache
The jars for the packages stored in: /home/codespace/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d7a341fe-c3df-4e76-b19d-fea884c0e725;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 in central
:: resolution report :: resolve 319ms :: artifacts dl 4ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apa

In [7]:

# Verify Spark session creation
spark.sql("SHOW DATABASES").show()


+---------+
|namespace|
+---------+
|  default|
+---------+



In [8]:

# Create an Iceberg table
spark.sql("""
  CREATE TABLE local.schema.users (
    id INT,
    name STRING,
    age INT
  ) USING iceberg""")

# Insert some sample data
spark.sql("""
  INSERT INTO local.schema.users VALUES
    (1, 'Alice', 30),
    (2, 'Bob', 25),
    (3, 'Charlie', 35)""")

# Query the data
result = spark.sql("SELECT * FROM local.schema.users")
result.show()

+---+-------+---+
| id|   name|age|
+---+-------+---+
|  1|  Alice| 30|
|  2|    Bob| 25|
|  3|Charlie| 35|
+---+-------+---+

